Run

In [0]:
%run ./config_secrets

Lógica de Autenticação e Bypass de Rede

In [0]:
import requests
import json
import random
import os


In [0]:
# No seu notebook config_secrets, defina o nome do catálogo que você acabou de criar
NOME_CATALOGO = "portfolio_mercado_livre"  # <- COLOQUE O NOME DO SEU CATÁLOGO AQUI

def obter_mapeamento_pipeline(user_id):
    return {
        "perfil": {
            "url": "https://api.mercadolibre.com/users/me",
            "tabela_landing": f"{NOME_CATALOGO}.landing.mercado_livre_perfil",
            "tabela_bronze": f"{NOME_CATALOGO}.bronze.mercado_livre_perfil"
        },
        "vendas": {
            "url": f"https://api.mercadolibre.com/orders/search?seller={user_id}",
            "tabela_landing": f"{NOME_CATALOGO}.landing.mercado_livre_vendas",
            "tabela_bronze": f"{NOME_CATALOGO}.bronze.mercado_livre_vendas"
        },
        "produtos": {
            "url": f"https://api.mercadolibre.com/users/{user_id}/items/search",
            "tabela_landing": f"{NOME_CATALOGO}.landing.mercado_livre_produtos",
            "tabela_bronze": f"{NOME_CATALOGO}.bronze.mercado_livre_produtos"
        }
    }

In [0]:
# Utiliza as variáveis que foram herdadas do notebook config_secrets
if MODO_SIMULADOR:
    print("🤖 Modo Simulador Ativo: Ignorando chamadas externas.")
    bearer_token = "MOCK_BEARER_TOKEN_12345"
    user_id = "999999999"
else:
    url_auth = "https://api.mercadolibre.com/oauth/token"
    url_completa = f"{url_auth}?grant_type=authorization_code&client_id={CLIENT_ID}&client_secret={CLIENT_SECRET}&code={CODE_DO_NAVEGADOR}&redirect_uri={REDIRECT_URI}"

    response_auth = requests.request("POST", url_completa, headers={"Content-Type": "application/json"}, data=json.dumps({"name": "Portfolio"}))
    if response_auth.status_code == 200:
        auth_data = response_auth.json()
        bearer_token = auth_data.get("access_token")
        user_id = auth_data.get("user_id")
    else:
        raise Exception(f"Falha crítica na autenticação ({response_auth.status_code}): {response_auth.text}")

print(f"✓ Token de Acesso gerado com sucesso para o Usuário: {user_id}")

Extração de Dados e Criação de Carga na Camada Landing (JSON Bruto)

In [0]:
# Força a criação das estruturas de banco de dados (Schemas) dentro do seu catálogo
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NOME_CATALOGO}.landing")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {NOME_CATALOGO}.bronze")

dados_brutos_pipeline = {}
endpoints_pipeline = obter_mapeamento_pipeline(user_id)

# 1. Simulação / Captura dos Payloads
if MODO_SIMULADOR:
    print("🤖 Alimentando payload de e-commerce sintético para simulação de API real...")
    dados_brutos_pipeline["perfil"] = {"id": user_id, "nickname": "ENG_DATA_SHOP", "site_id": "MLB"}
    
    lista_vendas_fakes = []
    estados = ["SP", "RJ", "MG", "RS", "PR", "BA", "SC"]
    for i in range(1, 1001):
        lista_vendas_fakes.append({
            "id": f"ORD-{100000 + i}",
            "date_created": f"2026-06-{random.randint(1,30)}T10:00:00.000-03:00",
            "total_amount": round(random.uniform(40.0, 1500.0), 2),
            "status": random.choice(["paid", "delivered"]),
            "buyer": {"id": f"BUY-{random.randint(4000, 8000)}", "shipping_address": {"state": random.choice(estados), "city": "Cidade Hub"}},
            "order_items": [{"item_id": f"MLB{random.randint(100, 150)}", "quantity": random.randint(1, 2), "unit_price": round(random.uniform(20.0, 500.0), 2)}]
        })
    dados_brutos_pipeline["vendas"] = {"results": lista_vendas_fakes}
    dados_brutos_pipeline["produtos"] = {"results": [f"MLB{id_prod}" for id_prod in range(100, 151)]}
else:
    for recurso, config in endpoints_pipeline.items():
        headers = {"Authorization": f"Bearer {bearer_token}", "Accept": "application/json"}
        response = requests.get(config["url"], headers=headers)
        if response.status_code == 200:
            dados_brutos_pipeline[recurso] = response.json()

# 2. CARGA NA LANDING SCHEMA: Salva o JSON bruto como String pura (Segurança e Auditoria do dado)
from pyspark.sql.functions import current_timestamp, lit

for recurso, dados in dados_brutos_pipeline.items():
    tabela_landing = endpoints_pipeline[recurso]["tabela_landing"]
    print(f"📦 Salvando payload original em formato JSON string na camada LANDING: {tabela_landing}")
    
    # Criamos um único registro contendo a String JSON inteira vinda da API
    payload_string = json.dumps(dados)
    df_landing = spark.createDataFrame([{"payload_bruto_json": payload_string}])
    
    # Adicionamos metadados mínimos na Landing
    df_landing_final = df_landing \
        .withColumn("dh_carga_landing", current_timestamp()) \
        .withColumn("nome_arquivo_origem", lit(f"api_extract_{recurso}.json"))
    
    # Salva no schema Landing (Formato Delta contendo JSON text)
    df_landing_final.write \
        .format("delta") \
        .mode("overwrite" if MODO_SIMULADOR else "append") \
        .saveAsTable(tabela_landing)

print("✓ Camada LANDING populada com sucesso no Unity Catalog!")

Leitura da Landing, Processamento Serverless e Carga na Bronze

In [0]:
for recurso in ["perfil", "vendas", "produtos"]:
    config = endpoints_pipeline[recurso]
    tabela_landing = config["tabela_landing"]
    tabela_bronze = config["tabela_bronze"]

    print(f" Lendo da LANDING e processando para a BRONZE: {recurso.upper()}")

    # Origem dos dados agora é a tabela física da Landing
    df_landing_source = spark.table(tabela_landing)

    # Coleta a string JSON de volta para processar as colunas
    payload_raw_str = df_landing_source.select("payload_bruto_json").first()[0]
    dados_originais = json.loads(payload_raw_str)

    if isinstance(dados_originais, dict) and "results" in dados_originais:
        lista_registros = dados_originais["results"]
    else:
        lista_registros = dados_originais if isinstance(dados_originais, list) else [dados_originais]

    lista_processada = []
    for registro in lista_registros:
        if not isinstance(registro, dict):
            registro_flat = {"id_item_referencia": registro}
        else:
            registro_flat = {ch: (json.dumps(vl) if isinstance(vl, (dict, list)) else (vl if vl is not None else "")) for ch, vl in registro.items()}
        lista_processada.append(registro_flat)

    # Processamento e Modelagem do DataFrame da Bronze
    df_bronze_estruturado = spark.createDataFrame(lista_processada)

    df_bronze_final = df_bronze_estruturado \
        .withColumn("dh_ingestao_etl", current_timestamp()) \
        .withColumn("origem_dados", lit(f"API_MERCADO_LIVRE_{recurso.upper()}"))

    # Carga na tabela Bronze definitiva do seu catálogo
    df_bronze_final.write \
        .format("delta") \
        .mode("append" if not MODO_SIMULADOR else "overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(tabela_bronze)

    print(f"✓ Carga concluída com sucesso na tabela: {tabela_bronze}")

print("\n ARQUITETURA MEDALLION INICIADA! Tabelas criadas nos Schemas LANDING e BRONZE.")